# LLM extraction — `alienation_alleged` via a local LLM, vs the rules baseline (RQ2)

The rules/lexicon extractor (`echr_extraction.ipynb`) measures **F1 = 0.507** on the frozen
120-case sample — transparent but weak. RQ2 (*can structured extraction move the capability
boundary, at what measured reliability?*) becomes a real experiment only as a **comparison**:

> same cases, same frozen gold labels, same metrics — **transparent rules vs local LLM**.

Design:
- **Model**: local Ollama (default `llama3.2`, 3B, CPU) — no paid APIs, reproducible offline.
- **Input per case**: every sentence containing an alienation-cluster term (the same lexicon
  the rules use), *with* its section label — but **without** the rules' context restriction,
  so the LLM can recover the false negatives the section filter causes.
- **Fast path**: a case with zero cluster mentions anywhere is `False` (conf 0.02) without an
  LLM call — identical to the rules' no-signal shortcut, and what keeps a CPU run feasible.
- **Scope**: `LABELED_ONLY=True` runs the 120 labelled cases (the evaluation); set it to
  `False` to build the full 719-row table overnight.
- **Checkpointing** every 10 cases → a killed run resumes.
- Output: `data/echr_extracted_llm.parquet`, report → `reports/llm_vs_rules_extraction.md`.

Labels are read **only** in the evaluation section — the extractor itself is label-free,
same contract as the rules extractor.

## 1. Configuration + reuse the deployed splitter/lexicon (exec from `echr_extraction.ipynb`)

In [ ]:
import json
import re
from pathlib import Path

DATA_DIR = Path("../data")
REPORT_DIR = Path("../reports")
ECHR_FILE  = DATA_DIR / "echr_parental_alienation.json"
LABELED_FILE = DATA_DIR / "echr_labeled_sample.csv"
RULES_TABLE  = DATA_DIR / "echr_extracted.parquet"
OUT_TABLE    = DATA_DIR / "echr_extracted_llm.parquet"
CHECKPOINT   = DATA_DIR / "echr_llm_extraction_checkpoint.json"
REPORT_OUT   = REPORT_DIR / "llm_vs_rules_extraction.md"

LLM_MODEL    = "llama3.2"     # local Ollama; 3B, CPU-feasible for ~100 short prompts
LABELED_ONLY = True            # True = the 120 labelled cases (evaluation); False = all 719
MAX_SENTS    = 10              # cluster sentences passed to the LLM per case
MAX_SENT_CHARS = 350
CHECKPOINT_EVERY = 10

ALLEGED_THRESHOLD = 0.50   # same threshold as echr_extraction.ipynb

# exec ONLY the splitter + lexicon cells (never the config cell — it would clobber the
# output paths above; that exact mistake once overwrote the rules table)
src_nb = json.loads(Path("echr_extraction.ipynb").read_text())
for marker in ["ECHR_ANCHORS = [", "CLUSTER = re.compile"]:
    cell = next("".join(c["source"]) for c in src_nb["cells"]
                if c["cell_type"] == "code" and marker in "".join(c["source"]))
    exec(cell)
assert OUT_TABLE.name == "echr_extracted_llm.parquet", "output path was clobbered by an exec'd cell"
print(f"reused splitter + lexicon from echr_extraction.ipynb | threshold {ALLEGED_THRESHOLD}")
print(f"model={LLM_MODEL} | labeled_only={LABELED_ONLY}")

section splitter ready | applicant-context sections: {'FACTS', 'unparsed', 'HEADER', 'PROCEDURE'}
alienation extractor ready (lexicon + attribution + negation)
reused splitter + lexicon from echr_extraction.ipynb | threshold 0.5
model=llama3.2 | labeled_only=True


## 2. Load cases + build per-case evidence pools

In [ ]:
import pandas as pd

records = json.loads(ECHR_FILE.read_text())
labels = pd.read_csv(LABELED_FILE) if LABELED_FILE.exists() else None
if LABELED_ONLY:
    keep = set(labels["id"]) if labels is not None else set()
    records = [r for r in records if r.get("itemid") in keep]
print(f"cases in scope: {len(records)}")


def evidence_pool(full_text):
    """All cluster-bearing sentences with their section label (no context restriction)."""
    secs = echr_sections(full_text or "")
    out = []
    for lab, t in secs:
        for s in _SENT.split(t):
            if CLUSTER.search(s):
                out.append((lab, s.strip()[:MAX_SENT_CHARS]))
            if len(out) >= MAX_SENTS:
                return out
    return out

pools = {r["itemid"]: evidence_pool(r.get("full_text", "")) for r in records}
n_empty = sum(1 for p in pools.values() if not p)
print(f"cluster-bearing cases: {len(pools) - n_empty} | no-mention fast path: {n_empty}")

cases in scope: 120
cluster-bearing cases: 83 | no-mention fast path: 37


## 3. The LLM extractor — strict-JSON prompt, temperature 0, checkpointed
The definition in the prompt is the **same confirmed definition** the hand-labelling used:
a *party's allegation* that the other parent alienated the child — not the court's own
narration, not mere estrangement as a fact, not the court's finding.

In [ ]:
PROMPT = """You classify excerpts from an ECHR family-law case.

Definition: alienation_alleged = true ONLY if a party to the case (typically the applicant,
or the other parent) ASSERTS/ALLEGES/COMPLAINS that the other parent alienated the child,
turned the child against them, manipulated or influenced the child against them.
It is false if the sentences merely describe estrangement as a fact, quote a statute,
report the court's own narration with no party attribution, or concern cultural/religious
estrangement.
The allegation counts even if the court later rejected it.

Sentences from the case (with the document section they appear in):
{sents}

Reply with ONLY a JSON object, no other text:
{{"alleged": true or false, "confidence": 0.0-1.0, "evidence": "<the single sentence that best supports your decision, verbatim>"}}"""

OLLAMA_OK = False
try:
    import ollama
    ollama.list()
    OLLAMA_OK = True
    print(f"Ollama reachable — model {LLM_MODEL}")
except Exception as e:
    print(f"!! Ollama unreachable ({e}) — start `ollama serve` and re-run this cell")

_JSON_RE = re.compile(r"\{.*\}", re.S)


def parse_reply(text):
    m = _JSON_RE.search(text or "")
    if not m:
        return None
    try:
        obj = json.loads(m.group(0))
        alleged = bool(obj.get("alleged"))
        conf = float(obj.get("confidence", 0.5))
        conf = min(1.0, max(0.0, conf))
        return dict(alleged=alleged, conf=conf, evidence=str(obj.get("evidence", ""))[:300])
    except (ValueError, TypeError):
        low = m.group(0).lower()
        if '"alleged": true' in low or "'alleged': true" in low:
            return dict(alleged=True, conf=0.6, evidence="")
        if '"alleged": false' in low:
            return dict(alleged=False, conf=0.4, evidence="")
        return None


def llm_extract(pool):
    sents = "\n".join(f"- [{lab}] {s}" for lab, s in pool)
    r = ollama.chat(model=LLM_MODEL,
                    messages=[{"role": "user", "content": PROMPT.format(sents=sents)}],
                    options={"temperature": 0})
    return parse_reply(r["message"]["content"])

print("llm_extract() ready")

Ollama reachable — model llama3.2
llm_extract() ready


## 4. Run (resumable) → `data/echr_extracted_llm.parquet`

In [ ]:
import time

done = json.loads(CHECKPOINT.read_text()) if CHECKPOINT.exists() else {}
print(f"checkpoint: {len(done)} cases already done")

if OLLAMA_OK:
    todo = [r for r in records if r["itemid"] not in done]
    t0 = time.time()
    for n, r in enumerate(todo, 1):
        iid = r["itemid"]
        pool = pools[iid]
        if not pool:
            done[iid] = dict(alleged=False, conf=0.02, evidence=None, prov="no-cluster")
        else:
            out = None
            for attempt in range(2):
                try:
                    out = llm_extract(pool)
                    if out:
                        break
                except Exception as e:
                    print(f"  {iid}: attempt {attempt+1} failed ({e})")
            if out is None:                      # unparseable twice -> abstain-style middle
                out = dict(alleged=False, conf=0.5, evidence=None)
                out["prov"] = "llm-unparseable"
            else:
                out["prov"] = f"llm:{LLM_MODEL};sents={len(pool)}"
            done[iid] = out
        if n % CHECKPOINT_EVERY == 0 or n == len(todo):
            CHECKPOINT.write_text(json.dumps(done))
            rate = (time.time() - t0) / n
            print(f"  {n}/{len(todo)} done ({rate:.1f}s/case, ~{rate*(len(todo)-n)/60:.0f} min left)")
    print(f"finished: {len(done)} cases")
else:
    print("skipped — Ollama unreachable")

if done:
    rows = [{"id": iid, "alienation_alleged_llm": d["alleged"],
             "alienation_conf_llm": d["conf"], "alienation_evidence_llm": d.get("evidence"),
             "provenance_llm": d.get("prov", "")} for iid, d in done.items()]
    llm_df = pd.DataFrame(rows)
    llm_df.to_parquet(OUT_TABLE, index=False)
    print(f"wrote {len(llm_df)} rows -> {OUT_TABLE.name}")
    print("positive rate:", round(llm_df.alienation_alleged_llm.mean(), 3))
else:
    llm_df = None

checkpoint: 120 cases already done
finished: 120 cases
wrote 120 rows -> echr_extracted_llm.parquet
positive rate: 0.342


## 5. Evaluation — rules vs LLM on the identical frozen gold labels
Same 120 cases, same gold, same threshold. Also: cross-validated ECE for the LLM's
self-reported confidence (same 5-fold protocol as `extraction_validation.ipynb`), and an
agreement breakdown showing *where* the two extractors differ.

In [ ]:
import numpy as np

if llm_df is None or labels is None:
    print("evaluation skipped (no LLM output or no labels)")
else:
    from sklearn.metrics import precision_recall_fscore_support, confusion_matrix
    from sklearn.isotonic import IsotonicRegression
    from sklearn.model_selection import StratifiedKFold

    rules = pd.read_parquet(RULES_TABLE)[["id", "alienation_alleged", "alienation_conf"]]
    ev = (labels.merge(llm_df, on="id", how="inner").merge(rules, on="id", how="inner"))
    ev["gold"] = pd.to_numeric(ev.gold_alienation_alleged, errors="coerce")
    ev = ev[ev.gold.notna()].copy()
    ev["gold"] = ev.gold.astype(int)
    y = ev.gold.to_numpy()
    print(f"evaluating on {len(ev)} labelled cases (gold positive rate {y.mean():.3f})\n")

    def score(name, pred):
        p, r, f1, _ = precision_recall_fscore_support(y, pred, average="binary", zero_division=0)
        tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
        print(f"{name:6s} P={p:.3f} R={r:.3f} F1={f1:.3f} | TP={tp} FP={fp} FN={fn} TN={tn}")
        return dict(precision=round(p, 3), recall=round(r, 3), f1=round(f1, 3),
                    tp=int(tp), fp=int(fp), fn=int(fn), tn=int(tn))

    m_rules = score("rules", ev.alienation_alleged.astype(int).to_numpy())
    m_llm   = score("LLM",   ev.alienation_alleged_llm.astype(int).to_numpy())

    # cross-validated ECE of the LLM's self-reported confidence
    def ece(conf, yy, n_bins=5):
        bins = np.linspace(0, 1, n_bins + 1)
        idx = np.clip(np.digitize(conf, bins) - 1, 0, n_bins - 1)
        tot = 0.0
        for b in range(n_bins):
            m = idx == b
            if m.sum():
                tot += m.sum() / len(yy) * abs(yy[m].mean() - conf[m].mean())
        return tot

    raw = ev.alienation_conf_llm.to_numpy(dtype=float)
    oof = np.full_like(raw, np.nan)
    for tr, te in StratifiedKFold(5, shuffle=True, random_state=42).split(raw.reshape(-1, 1), y):
        iso = IsotonicRegression(out_of_bounds="clip", y_min=0, y_max=1)
        iso.fit(raw[tr], y[tr])
        oof[te] = iso.predict(raw[te])
    ece_raw, ece_oof = ece(raw, y.astype(float)), ece(oof, y.astype(float))
    print(f"\nLLM confidence: ECE raw={ece_raw:.3f} | ECE OOF-calibrated={ece_oof:.3f}")

    # where do the extractors differ?
    pr = ev.alienation_alleged.astype(bool)
    pl = ev.alienation_alleged_llm.astype(bool)
    g = ev.gold.astype(bool)
    print("\nagreement breakdown:")
    print(f"  both right : {int(((pr == g) & (pl == g)).sum())}")
    print(f"  LLM only   : {int(((pr != g) & (pl == g)).sum())}   <- what the LLM adds")
    print(f"  rules only : {int(((pr == g) & (pl != g)).sum())}   <- what the LLM loses")
    print(f"  both wrong : {int(((pr != g) & (pl != g)).sum())}")

    print("\nLLM-only wins (rules wrong, LLM right):")
    for _, r0 in ev[(pr != g) & (pl == g)].head(8).iterrows():
        print(f"  {r0.id} gold={r0.gold} | {str(r0.title)[:60]}")

evaluating on 120 labelled cases (gold positive rate 0.333)

rules  P=0.543 R=0.475 F1=0.507 | TP=19 FP=16 FN=21 TN=64
LLM    P=0.585 R=0.600 F1=0.593 | TP=24 FP=17 FN=16 TN=63

LLM confidence: ECE raw=0.260 | ECE OOF-calibrated=0.001

agreement breakdown:
  both right : 69
  LLM only   : 18   <- what the LLM adds
  rules only : 14   <- what the LLM loses
  both wrong : 19

LLM-only wins (rules wrong, LLM right):
  001-191488 gold=1 | CASE OF BOGONOSOVY v. RUSSIA
  001-220479 gold=1 | X AND OTHERS v. SLOVENIA and 1 other application
  001-232004 gold=0 | CASE OF TZIOUMAKA v. GREECE
  001-218132 gold=1 | CASE OF JURIŠIĆ v. CROATIA (No. 2)
  001-237297 gold=1 | A.P. AND A.M. v. THE CZECH REPUBLIC
  001-177079 gold=1 | CASE OF SEVERE v. AUSTRIA
  001-189774 gold=1 | D.D.F. AND OTHERS v. ROMANIA
  001-164917 gold=0 | CASE OF E.S. v. ROMANIA AND BULGARIA


## 6. Report → `reports/llm_vs_rules_extraction.md`

In [ ]:
if llm_df is not None and labels is not None:
    lines = ["# RQ2 experiment — rules vs local-LLM extraction of `alienation_alleged`\n\n"]
    lines.append(f"- identical frozen gold sample (n={len(ev)}), identical threshold "
                 f"({ALLEGED_THRESHOLD}); LLM = `{LLM_MODEL}` via local Ollama, temperature 0, "
                 "CPU-only, no paid APIs.\n"
                 "- LLM input = all cluster-lexicon sentences with section labels (no context "
                 "restriction); cases without any cluster mention short-circuit to False for "
                 "both extractors.\n\n")
    lines.append("| extractor | precision | recall | F1 | TP | FP | FN | TN |\n"
                 "|---|---|---|---|---|---|---|---|\n")
    lines.append(f"| rules (transparent) | {m_rules['precision']} | {m_rules['recall']} | "
                 f"**{m_rules['f1']}** | {m_rules['tp']} | {m_rules['fp']} | {m_rules['fn']} | {m_rules['tn']} |\n")
    lines.append(f"| LLM ({LLM_MODEL}) | {m_llm['precision']} | {m_llm['recall']} | "
                 f"**{m_llm['f1']}** | {m_llm['tp']} | {m_llm['fp']} | {m_llm['fn']} | {m_llm['tn']} |\n\n")
    lines.append(f"- LLM self-reported confidence: ECE raw = **{ece_raw:.3f}**, "
                 f"5-fold OOF-calibrated = **{ece_oof:.3f}** (same protocol as the rules "
                 "extractor's calibration).\n")
    lines.append(f"- agreement: both right {int(((pr == g) & (pl == g)).sum())}, "
                 f"LLM-only right {int(((pr != g) & (pl == g)).sum())}, "
                 f"rules-only right {int(((pr == g) & (pl != g)).sum())}, "
                 f"both wrong {int(((pr != g) & (pl != g)).sum())}.\n\n")
    lines.append("## Reading\n"
                 "- This is the RQ2 result: whether moving from transparent rules to a local "
                 "LLM buys reliability, and how much auditability it costs (the LLM returns a "
                 "verbatim evidence sentence, but its decision process is opaque).\n"
                 "- Both extractors share the gold sample's limitations (single annotator, "
                 "anchored template — see `extraction_validation_report.md`).\n"
                 "- Domain of validity: ECHR Article 8 contact/alienation cases only.\n")
    REPORT_OUT.write_text("".join(lines), encoding="utf-8")
    print("wrote", REPORT_OUT)
else:
    print("report skipped")

wrote ../reports/llm_vs_rules_extraction.md
